# Landmarks de face, contorno e fundo

In [32]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

from retinaface import RetinaFace
import tensorflow as tf
import json
import cv2
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import ipywidgets as widgets
from matplotlib import animation
from IPython.display import HTML, Video, display
import torch

def check_gpu():
    torch.cuda.todevice(0)  # Tenta acessar a GPU
    if torch.cuda.is_available():
        print(f"GPU ativa: {torch.cuda.get_device_name(0)}")
        return True
    else:
        print("Rodando em CPU")
        return False

tf.get_logger().setLevel("ERROR")

In [33]:
check_gpu()

Rodando em CPU


False

In [24]:
def load_video_frames(video_path, max_frames=None):
    cap = cv2.VideoCapture(video_path)

    frames = []
    count = 0

    while True:
        ret, frame = cap.read()

        if not ret:
            break

        frames.append(frame)

        count += 1
        if max_frames and count >= max_frames:
            break

    cap.release()

    return np.array(frames)

In [25]:
def standardize_frame(frame, max_size=640):
    h, w = frame.shape[:2]

    scale = min(max_size / max(h, w), 1.0)
    new_w, new_h = int(w * scale), int(h * scale)

    frame_resized = cv2.resize(frame, (new_w, new_h), interpolation=cv2.INTER_AREA)

    return frame_resized, scale

In [26]:
def detect_face(frame):
    detections = RetinaFace.detect_faces(frame)

    if len(detections) == 0:
        return None

    # pega a face com maior score
    face = list(detections.values())[0]

    bbox = face["facial_area"]
    landmarks = face["landmarks"]

    return {
        "bbox": bbox,
        "landmarks": landmarks,
        "score": face["score"]
    }

In [27]:
def align_face(frame, landmarks, output_size=256):

    left_eye = landmarks["left_eye"]
    right_eye = landmarks["right_eye"]

    # ângulo
    dy = right_eye[1] - left_eye[1]
    dx = right_eye[0] - left_eye[0]
    angle = np.degrees(np.arctan2(dy, dx))

    eyes_center = (
        (left_eye[0] + right_eye[0]) // 2,
        (left_eye[1] + right_eye[1]) // 2
    )

    M = cv2.getRotationMatrix2D(eyes_center, angle, 1.0)

    aligned = cv2.warpAffine(frame, M, (frame.shape[1], frame.shape[0]))

    return aligned, M

In [28]:
def create_face_regions(frame, bbox, padding=0.2):
    h, w = frame.shape[:2]

    x1, y1, x2, y2 = bbox

    bw = x2 - x1
    bh = y2 - y1

    # aplica padding
    px = int(bw * padding)
    py = int(bh * padding)

    x1p = max(0, x1 - px)
    y1p = max(0, y1 - py)
    x2p = min(w, x2 + px)
    y2p = min(h, y2 + py)

    # máscara face interna
    face_mask = np.zeros((h, w), dtype=np.uint8)
    face_mask[y1:y2, x1:x2] = 1

    # máscara expandida
    expanded_mask = np.zeros((h, w), dtype=np.uint8)
    expanded_mask[y1p:y2p, x1p:x2p] = 1

    # borda = expandida - interna
    border_mask = expanded_mask - face_mask

    # fundo = resto
    background_mask = 1 - expanded_mask

    return {
        "face": face_mask,
        "border": border_mask,
        "background": background_mask,
        "bbox_expanded": (x1p, y1p, x2p, y2p)
    }

## Debugging de regiões

In [29]:
# Desenha a caixa delimitadora no frame
def draw_face_box(frame, bbox, color=(0, 255, 0), thickness=2):
    x1, y1, x2, y2 = bbox
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, thickness)
    return frame


def overlay_regions(frame, regions):
    overlay = frame.copy()

    face_mask = regions["face"]
    border_mask = regions["border"]
    background_mask = regions["background"]

    overlay[face_mask == 1] = (0, 255, 0)       # verde
    overlay[border_mask == 1] = (0, 0, 255)     # vermelho
    overlay[background_mask == 1] = (255, 0, 0) # azul

    alpha = 0.3
    return cv2.addWeighted(overlay, alpha, frame, 1 - alpha, 0)


# Visualiza o vídeo com as caixas e regiões sobrepostas
def play_video_with_boxes(video_path, interval=40, max_frames=120):
    gpu = check_gpu()
    if gpu:
        print("\nUsando GPU para detecção")
    else: print("\nUsando CPU para detecção")

    frames = load_video_frames(video_path)

    if len(frames) == 0:
        raise ValueError("Sem frames")

    if len(frames) > max_frames:
        idx = np.linspace(0, len(frames) - 1, int(max_frames)).astype(int)
        frames = frames[idx]

    detections = []
    regions_list = []

    tracker = None
    last_bbox = None

    DETECT_EVERY = 5 

    print("\nProcessando frames...")

    for i, frame in enumerate(frames):

        frame_std, scale = standardize_frame(frame)

        use_detection = (i % DETECT_EVERY == 0) or (tracker is None)

        bbox_original = None

        # detecção a cada N frames ou se o tracker falhar
        if use_detection:
            det = detect_face(frame_std)

            if det is not None:
                bbox_scaled = det["bbox"]
                bbox_original = [int(x / scale) for x in bbox_scaled]

                # inicializa tracker
                tracker = cv2.TrackerCSRT_create()
                tracker.init(frame, tuple(bbox_original))

                last_bbox = bbox_original

        # faz o tracking nos frames intermediários
        else:
            success, tracked_box = tracker.update(frame)

            if success:
                x, y, w, h = map(int, tracked_box)
                bbox_original = [x, y, x + w, y + h]
                last_bbox = bbox_original
            else:
                bbox_original = None

        # fallbacks caso detecção e tracking falhem
        if bbox_original is None:

            if last_bbox is not None:
                bbox_original = last_bbox

            else:
                # fallback: centro da imagem
                h, w = frame.shape[:2]
                cx, cy = w // 2, h // 2
                size = min(h, w) // 4

                bbox_original = [
                    cx - size, cy - size,
                    cx + size, cy + size
                ]

        # extraindo as regiões para visualização
        regions = create_face_regions(frame, bbox_original)

        detections.append(bbox_original)
        regions_list.append(regions)

    print("\nProcessamento concluído")

    # visualização
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.axis("off")

    img = ax.imshow(cv2.cvtColor(frames[0], cv2.COLOR_BGR2RGB))

    def update(i):
        frame = frames[i].copy()

        bbox = detections[i]
        regions = regions_list[i]

        frame = draw_face_box(frame, bbox)
        frame = overlay_regions(frame, regions)

        img.set_data(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        return [img]

    ani = animation.FuncAnimation(
        fig,
        update,
        frames=len(frames),
        interval=interval,
        blit=True,
    )

    plt.close(fig)
    display(HTML(ani.to_jshtml()))

In [30]:
df = pd.read_csv("C:\\Users\\Guilherme Monteiro\\Desktop\\TCC\\data\\video-metadata-publish-with-links.csv")
df['video_path'] = df['Filename'].apply(lambda x: "C:\\Users\\Guilherme Monteiro\\Desktop\\TCC\\data\\videos\\" + x)

In [31]:
play_video_with_boxes(df['video_path'].iloc[0])

Rodando em CPU

Usando CPU para detecção

Processando frames...


KeyboardInterrupt: 